# The fill search

[Notebook 1](01-building-a-grid.ipynb) built a puzzle end to end and treated
the fill as one step: hand it a pattern, get back a grid full of words. This is
what happens inside that step.

The pattern never changes here. Only letters are assigned, so anything that
goes wrong is unambiguously the search's fault rather than the grid's.

1. What the search is actually choosing between
2. One node, in detail: forward checking and which slot to expand
3. Watching a real search
4. Choosing *which* word, not just a legal one
5. Aiming at a familiarity rather than maximising it
6. When it fails, and what restarts are worth

In [1]:
import math
import os
import sys
import time
from collections import Counter

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

from crossword import library
from crossword.fill import Filler
from crossword.index import Index
from crossword.render import show
from crossword.rules import RuleSet
from crossword.words import load

entries = load("crossword/UKACD.txt")
scores = {}
with open("crossword/scores.txt", encoding="utf-8") as handle:
    for line in handle:
        if not line.startswith("#") and line.strip():
            word, value = line.split()
            scores[word] = float(value)

index = Index(entries, scores)
print(f"{len(entries):,} words, {len(scores):,} of them scored for familiarity")

221,835 words, 105,373 of them scored for familiarity


## 1. What is being chosen

A pattern of 30 entries, each needing a word. The entries are not independent:
every checked cell belongs to two of them at once, so a choice in one entry
constrains a crossing entry, which constrains the entries crossing *that*.

The size of the raw problem is not the interesting number, but it is worth
seeing once.

In [2]:
pattern = library.load()[0]
grid = pattern.grid()
slots = grid.slots()

total = 0.0
for slot in slots:
    total += math.log10(len(index[slot.length].words))
print(f"{len(slots)} entries")
print(f"ways to fill them ignoring every crossing: about 10^{total:.0f}")
print(f"checked cells, where two entries must agree: {len(grid.checked_cells())}")
show(grid)

30 entries
ways to fill them ignoring every crossing: about 10^128
checked cells, where two entries must agree: 58


## 2. One node

At every node the filler does a single pass over the unfilled entries and, for
each, computes the set of words still fitting it. That one pass does both jobs
the search needs, which is why they are not written as separate steps:

- **forward check** — if any entry has no candidates, this branch is already
  dead and no word need be tried at all
- **ordering** — expand the entry with the *fewest* candidates first, the
  minimum-remaining-values heuristic

Expanding the most constrained entry first is what keeps the tree narrow. The
alternative, taking entries in reading order, spends its early choices on
entries that had thousands of options and discovers the impossible corner only
after a great deal of work.

Here is that pass on the empty grid.

In [3]:
filler = Filler(grid, index, RuleSet(), seed=0, commonness=3.0, aim=0.85)

counts = []
for slot in slots:
    mask = filler._candidates(slot)
    counts.append((bin(mask).count("1"), slot))
counts.sort(key=lambda pair: pair[0])   # Slot is frozen but not orderable

print("fewest candidates first:")
for count, slot in counts[:4]:
    print(f"  {count:6,}  {slot.direction:6} at ({slot.row:2},{slot.col:2}) "
          f"length {slot.length}")
print("  ...")
for count, slot in counts[-2:]:
    print(f"  {count:6,}  {slot.direction:6} at ({slot.row:2},{slot.col:2}) "
          f"length {slot.length}")

print(f"\nMRV expands the {counts[0][1].length}-letter entry with "
      f"{counts[0][0]:,} candidates,")
print(f"not the {counts[-1][1].length}-letter one with {counts[-1][0]:,}.")

fewest candidates first:
   4,582  across at ( 4,11) length 4
   4,582  across at (10, 0) length 4
   4,582  down   at (11, 4) length 4
   4,582  down   at ( 0,10) length 4
  ...
  33,130  down   at ( 6, 6) length 9
  33,130  down   at ( 0, 8) length 9

MRV expands the 4-letter entry with 4,582 candidates,
not the 9-letter one with 33,130.


### How fast the candidates collapse

Placing one word does not constrain one entry. It constrains every entry
crossing it, and on a British lattice that is several at once.

In [4]:
first = counts[0][1]
word_id = index[first.length].ids(filler._candidates(first))[0]
word = index[first.length].words[word_id]
written = filler._place(first, word, word_id)
print(f"placed {word.upper()} in {first.direction} at "
      f"({first.row},{first.col})\n")

print("entry                          before     after")
for count, slot in counts:
    if slot is first:
        continue
    after = bin(filler._candidates(slot)).count("1")
    if after != count:
        print(f"  {slot.direction:6} ({slot.row:2},{slot.col:2}) len {slot.length:2}"
              f"       {count:8,}  {after:8,}")
filler._unplace(first, word_id, written)

placed ABAC in across at (4,11)

entry                          before     after
  across (10, 0) len  4          4,582     4,581
  down   (11, 4) len  4          4,582     4,581
  down   ( 0,10) len  4          4,582     4,581
  down   ( 0,12) len  8         32,192       908
  down   ( 0,14) len  8         32,192     1,167


## 3. Watching a real search

`Stats` records what the search did. `nodes` counts choices made, `backtracks`
counts choices withdrawn, and `dead_slot` counts how often each entry was the
one found with nothing left in it.

In [5]:
grid = pattern.grid()
filler = Filler(grid, index, RuleSet(), seed=3, commonness=3.0, aim=0.85)
began = time.time()
ok = filler.fill()
print(f"filled: {ok}   {filler.stats}")
print(f"entries: {len(grid.slots())}, all lettered: "
      f"{len(grid.letters) == grid.size ** 2 - len(grid.blocks)}")
show(grid)

filled: True   nodes 33  backtracks 3  restarts 0  0.03s
entries: 30, all lettered: True


A grid of this shape usually needs only a few hundred nodes. The number worth
noticing is how close `backtracks` sits to `nodes`: nearly every choice made is
eventually withdrawn, and the search still finishes quickly, because withdrawing
a choice is cheap and the forward check kills bad branches before any word is
enumerated.

## 4. Choosing which word

Every candidate at a node is legal. Legality is not the difficulty — there are
thousands of legal words for most entries, and taking the first one gives a
grid full of words no solver has met.

So candidates are *weighted* rather than filtered. The order is Gumbel-top-k:
add Gumbel noise to each word's log weight and sort. That is exactly weighted
sampling without replacement, and the important property is that nothing is
excluded. An obscure word stays reachable when the crossings leave nothing
else, which is the difference between a grid that reads well and a grid that
fails to fill at all.

`commonness` is how hard that preference pulls. Compare the two extremes on the
same pattern and seed.

In [6]:
def fill_once(commonness, aim=0.85, seed=7):
    made = pattern.grid()
    worker = Filler(made, index, RuleSet(), seed=seed,
                    commonness=commonness, aim=aim)
    assert worker.fill(), "did not fill"
    return made


def familiarity(made):
    ranks = []
    for slot in made.slots():
        bucket = index.lengths[slot.length]
        word_id = bucket.by_word.get(made.pattern(slot))
        if word_id is not None and bucket.quantile:
            ranks.append(bucket.quantile[word_id])
    return sum(ranks) / len(ranks)


for setting in (0.0, 3.0):
    made = fill_once(setting)
    words = sorted((made.pattern(s) for s in made.slots()), key=len)
    print(f"commonness {setting}: familiarity {familiarity(made):.2f}")
    print(f"   {', '.join(w.upper() for w in words[:8])}")

commonness 0.0: familiarity 0.52
   NICK, ELTS, SLUG, RARE, VOULU, THUMP, LUFFA, ORGIA
commonness 3.0: familiarity 0.76
   FIRE, IBIS, GARB, LIMP, IDIOM, GRILL, WAEFU, THIEF


## 5. Aiming, not maximising

The obvious way to use a familiarity score is to prefer the most familiar word
available. That produces grids of ISLE, OVER and AREA: technically ordinary,
and nothing like a published puzzle.

Real answers are not maximally familiar. Measured against Guardian answers,
they sit at a steady *rank within their own length* — about 0.86, and roughly
that at every length. Rank matters rather than raw frequency, because long
words are rarer by nature and their raw scores cannot be compared with short
ones.

So `aim` is a target, not a ceiling, and the weight is the negative distance
from it. Words far more obscure than the target are pushed down; so are words
far more obvious.

In [7]:
bucket = index.lengths[5]
ranked = sorted(range(len(bucket.words)), key=lambda i: bucket.quantile[i])

for label, quantile in (("most obscure", 0.0), ("aim 0.85", 0.85),
                        ("most ordinary", 1.0)):
    at = min(int(quantile * (len(ranked) - 1)), len(ranked) - 1)
    low = min(max(0, at - 3), len(ranked) - 7)
    around = [bucket.words[i].upper() for i in ranked[low:low + 7]]
    print(f"  {label:14} {', '.join(around)}")

  most obscure   ABACS, ABLET, ABODY, ABOIL, ABORE, ABRAY, ABSEY
  aim 0.85       NORMS, PINCH, QUEER, ROVER, SCARS, SYRUP, TONES
  most ordinary  FIRST, OTHER, WOULD, WHICH, THERE, THEIR, ABOUT


In [8]:
print("effect on a whole grid:\n")
print("aim   familiarity   a sample of the fill")
for aim in (0.5, 0.85, 1.0):
    made = fill_once(3.0, aim=aim)
    words = [made.pattern(s) for s in made.slots() if s.length <= 5]
    print(f"{aim:4}  {familiarity(made):9.2f}   "
          f"{', '.join(w.upper() for w in sorted(words)[:7])}")
print("\npublished Guardian answers average 0.86")

effect on a whole grid:

aim   familiarity   a sample of the fill
 0.5       0.51   ACHY, CHAS, CHYME, IVIED, LIER, MOPUP, PILAU
0.85       0.76   FIRE, GARB, GRILL, IBIS, IDIOM, LIMP, THIEF
 1.0       0.88   ACRE, DOIN, FIRE, KESAR, MENU, OOMPH, OUGHT

published Guardian answers average 0.86


## 6. Failure, and what restarts are worth

When the search cannot finish it exhausts its node budget. `dead_slot` then
says where the trouble was: an entry appearing thousands of times is one the
search kept arriving at with nothing left to put in it.

Restarts are the standard remedy, and whether they help is a property of the
problem rather than of the search. They help when the runtime is heavily
tailed — when a lucky start finishes quickly and an unlucky one never will.
They do nothing when every attempt fails the same way.

This project has measured both cases:

| problem | restarts |
|---|---|
| 6x6 double word square | 60 tries of 8,000 nodes found one in 39s; 6 tries of 200,000 took 209s |
| 7x7 double word square | nothing at any granularity from 5x600,000 to 600x5,000 |
| barred grids that will not fill | 240,000 nodes as 4 searches or as 480 gives 0 of 8 either way |

The first is a tail worth chasing. The other two are not: when success is
fast and failure is total, a bigger budget rescues almost nothing, and the
answer lies in the pattern or in the search itself rather than in the seed.

In [9]:
# A 7x7 double word square: seven across words and seven down words, every
# letter checked twice.  This one is known not to come out -- seventeen million
# nodes have been spent on it -- so it is a good look at what failure does.
from crossword.grid import Grid

square = Grid(size=7)
square_rules = RuleSet(min_entry_length=7, min_checked_fraction=1.0,
                       max_consecutive_unchecked=0, symmetry="none")
seven = Index([e for e in entries
               if len(e.text) == 7 and not e.phrase], scores)

worker = Filler(square, seven, square_rules, seed=1,
                node_budget=30000, commonness=0.0)
began = time.time()
print(f"filled: {worker.fill(restarts=1)}   ({time.time() - began:.1f}s)")
print(worker.stats)

worst = sorted(worker.stats.dead_slot.items(), key=lambda kv: -kv[1])[:5]
dead = sum(worker.stats.dead_slot.values())
print(f"\n{dead:,} of {worker.stats.nodes:,} nodes ended on an empty entry.")
print("the entries it kept arriving at with nothing left:")
for (direction, row, col), hits in worst:
    print(f"  {direction:6} at ({row},{col})  {hits:,} times")

filled: False   (2.4s)
nodes 30001  backtracks 29995  restarts 0  2.41s

25,199 of 30,001 nodes ended on an empty entry.
the entries it kept arriving at with nothing left:
  across at (1,0)  11,583 times
  down   at (0,0)  3,461 times
  across at (2,0)  2,550 times
  across at (3,0)  2,001 times
  down   at (0,1)  1,538 times


## Next

- **the other styles**: American and barred, and what had to change for each
- **ninas and pangrams**: constraints a setter adds on top

Back to [notebook 1](01-building-a-grid.ipynb). `BASELINE.md` records what each
change to the search was measured to be worth, including the ones that were
reverted.